In [ ]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.datasets import load_iris
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score

In [ ]:
class AdaBoost:
    def __init__(self, n_estimators=50):
        self.n_estimators = n_estimators
        self.alphas = []
        self.models = []

    def fit(self, X, y):
        n_samples, n_features = X.shape  
        w = np.ones(n_samples) / n_samples 

        for _ in range(self.n_estimators):
            model = DecisionTreeClassifier(max_depth=1)  
            model.fit(X, y, sample_weight=w)  
            predictions = model.predict(X)  

            err = np.sum(w * (predictions != y)) / np.sum(w)

            alpha = 0.5 * np.log((1 - err) / (err + 1e-10))

            self.models.append(model) 
            self.alphas.append(alpha)  

            w *= np.exp(-alpha * y * predictions)  
            w /= np.sum(w)


        def predict(self, X):
        strong_preds = np.zeros(X.shape[0])  

        for model, alpha in zip(self.models, self.alphas):
            predictions = model.predict(X)  
            strong_preds += alpha * predictions  

        return np.sign(strong_preds).astype(int)

In [ ]:
import numpy as np

class DecisionStump:
    """
    A simple decision stump (weak learner) for binary classification.
    Splits data based on one feature and a threshold.
    """
    def __init__(self):
        self.feature_index = None
        self.threshold = None
        self.polarity = 1
        self.alpha = None  # Learner weight

    def predict(self, X):
        """Predict class labels for samples in X."""
        n_samples = X.shape[0]
        predictions = np.ones(n_samples)
        if self.polarity == 1:
            predictions[X[:, self.feature_index] < self.threshold] = -1
        else:
            predictions[X[:, self.feature_index] > self.threshold] = -1
        return predictions


class AdaBoost:
    """
    AdaBoost classifier using decision stumps as weak learners.
    """
    def __init__(self, n_estimators=50):
        if not isinstance(n_estimators, int) or n_estimators <= 0:
            raise ValueError("n_estimators must be a positive integer.")
        self.n_estimators = n_estimators
        self.stumps = []

    def fit(self, X, y):
        """
        Train AdaBoost classifier.
        X: numpy array of shape (n_samples, n_features)
        y: numpy array of shape (n_samples,), labels must be -1 or 1
        """
        X = np.array(X, dtype=float)
        y = np.array(y, dtype=float)

        if set(np.unique(y)) != {-1.0, 1.0}:
            raise ValueError("Labels must be -1 or 1.")

        n_samples, n_features = X.shape
        # Initialize weights equally
        w = np.full(n_samples, 1 / n_samples)

        for _ in range(self.n_estimators):
            stump = DecisionStump()
            min_error = float("inf")

            # Find the best stump
            for feature_i in range(n_features):
                feature_values = X[:, feature_i]
                thresholds = np.unique(feature_values)
                for threshold in thresholds:
                    for polarity in [1, -1]:
                        predictions = np.ones(n_samples)
                        if polarity == 1:
                            predictions[feature_values < threshold] = -1
                        else:
                            predictions[feature_values > threshold] = -1

                        # Weighted error
                        error = np.sum(w[y != predictions])

                        if error < min_error:
                            min_error = error
                            stump.feature_index = feature_i
                            stump.threshold = threshold
                            stump.polarity = polarity

            # Avoid division by zero
            epsilon = 1e-10
            stump.alpha = 0.5 * np.log((1 - min_error) / (min_error + epsilon))

            # Update weights
            predictions = stump.predict(X)
            w *= np.exp(-stump.alpha * y * predictions)
            w /= np.sum(w)  # Normalize

            self.stumps.append(stump)

    def predict(self, X):
        """Predict class labels for samples in X."""
        X = np.array(X, dtype=float)
        stump_preds = [stump.alpha * stump.predict(X) for stump in self.stumps]
        y_pred = np.sign(np.sum(stump_preds, axis=0))
        return y_pred


# Example usage
if __name__ == "__main__":
    # Simple dataset
    X_train = np.array([
        [1, 2],
        [2, 3],
        [3, 1],
        [4, 3],
        [5, 2]
    ])
    y_train = np.array([1, 1, -1, -1, 1])  # Labels must be -1 or 1

    model = AdaBoost(n_estimators=5)
    model.fit(X_train, y_train)

    predictions = model.predict(X_train)
    print("Predictions:", predictions)
    print("Actual:     ", y_train)
